In [1]:
from setuptools.dist import sequence

import utils
import numpy as np
import pandas as pd
import os
import random
from scipy.stats import chisquare

import matplotlib.pyplot as plt
import seaborn as sns # Loads in the theme
sns.set_theme()

pd.set_option('future.no_silent_downcasting', True)
pd.set_option('display.max_rows', None)

In [2]:
# Make output folders
output_dir = os.path.join('..','pre-processed', 'data_for_fMRI_modelling')
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

In [3]:
# Preparation
experiment_output_path = os.path.join('..', '..', 'phd_1_task', 'experiment_output', 'behavioural_files')

In [4]:
# Load the path and read the CSV
subject_id = 820
session_number = 1

sequence_path = [os.path.join('..', '..', 'phd_1_task', 'sequences', f'parameters_per_stimuli_sub-{subject_id:003}_session-{session_number:02}.csv')]
sequence_dataframe = pd.read_csv(sequence_path[0], sep=';')

# To ensure compatibility with the analysis_functions, we will make subject_id and session_number columns
sequence_dataframe.insert(0, 'subject_id', subject_id)
sequence_dataframe.insert(1, 'session', session_number)

# Make up responses
dataframe_height = len(sequence_dataframe['stimuli_type'])
made_up_response_delays = np.linspace(0.4, 1.15, dataframe_height).tolist()
## Then shuffle it and add it to the dataframe
random.shuffle(made_up_response_delays)
sequence_dataframe['RT_s'] = made_up_response_delays

sequence_dataframe['emotional_cue_time'] = sequence_dataframe['iti'].shift(fill_value=0).cumsum()
sequence_dataframe['response_time'] = sequence_dataframe['emotional_cue_time'] + sequence_dataframe['RT_s']

print(sequence_dataframe)

     subject_id  session  trial  stimuli_type  probability_condition  \
0           820        1      0             2                     50   
1           820        1      1             2                     50   
2           820        1      2             1                     50   
3           820        1      3             1                     50   
4           820        1      4             2                     50   
5           820        1      5             1                     50   
6           820        1      6             1                     50   
7           820        1      7             2                     50   
8           820        1      8             2                     80   
9           820        1      9             2                     80   
10          820        1     10             1                     20   
11          820        1     11             1                     20   
12          820        1     12             2                   

# Check congruency, valence and volatility

In [5]:
# Check a block's congruency
sequence_dataframe['congruency'] = sequence_dataframe.apply(utils.analysis_functions.check_congruency, args=(True,), axis=1)

# Check a block's valence
sequence_dataframe['valence'] = sequence_dataframe.apply(utils.analysis_functions.check_valence, args=(True,), axis=1)
print(sequence_dataframe[['probability_condition','valence','congruency']])

     probability_condition  valence  congruency
0                       50        1          -1
1                       50        1          -1
2                       50        0          -1
3                       50        0          -1
4                       50        1          -1
5                       50        0          -1
6                       50        0          -1
7                       50        1          -1
8                       80        1           0
9                       80        1           0
10                      20        0           0
11                      20        0           0
12                      80        1           0
13                      80        1           0
14                      20        0           0
15                      80        1           0
16                      20        0           0
17                      20        0           0
18                      20        0           0
19                      20        0     

In [6]:
# Check a block's volatility
sequence_dataframe = sequence_dataframe[sequence_dataframe['probability_condition'] != 50]
sequence_dataframe = sequence_dataframe.groupby('stimuli_type', group_keys=False).apply(lambda x: utils.analysis_functions.check_volatility(x, True))
sequence_dataframe = sequence_dataframe.sort_values(by=['stimuli_type','trial'], ascending=[True, True])
print(sequence_dataframe[['stimuli_type','volatility','probability_condition']])

     stimuli_type volatility  probability_condition
0               1          0                     20
1               1          0                     20
2               1          0                     20
3               1          0                     20
4               1          0                     20
5               1          0                     20
6               1          0                     20
7               1          0                     20
8               1          0                     20
9               1          0                     20
10              1          0                     20
11              1          0                     20
12              1          0                     20
13              1          0                     20
14              1          0                     20
15              1          0                     20
16              1          0                     20
17              1          0                     20
18          

/var/folders/bh/ld02p1_550d75y0x32dmpb540000gp/T/ipykernel_6198/4173453293.py:3: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  sequence_dataframe = sequence_dataframe.groupby('stimuli_type', group_keys=False).apply(lambda x: utils.analysis_functions.check_volatility(x, True))


In [7]:
sequence_dataframe = sequence_dataframe.sort_values(by=['subject_id', 'session', 'stimuli_type', 'trial'], ascending=[True, True, True, True])
sequence_dataframe.reset_index(drop=True, inplace=True)
print(sequence_dataframe[['subject_id', 'session', 'congruency', 'valence', 'volatility']])
print(sequence_dataframe.shape[0])

     subject_id  session  congruency  valence volatility
0           820        1           0        0          0
1           820        1           0        0          0
2           820        1           0        0          0
3           820        1           0        0          0
4           820        1           0        0          0
5           820        1           0        0          0
6           820        1           0        0          0
7           820        1           0        0          0
8           820        1           0        0          0
9           820        1           0        0          0
10          820        1           0        0          0
11          820        1           0        0          0
12          820        1           0        0          0
13          820        1           0        0          0
14          820        1           0        0          0
15          820        1           0        0          0
16          820        1       

# Now export the data

In [8]:
sequence_dataframe.to_csv(f'{output_dir}/combined_dataframe_binary_pilot_hard.csv', sep=';', index_label='index')